In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [ ]:
fecha_mes_base='2026-07-01'
servidor_01=76
tipi_cond1='SAE'
tipi_cond2='SAR'
tipi_cond3='AGENDA'
tb_tipolofia='tTipologia_Cencosud_PPFF's
tipi_cod='Codigo'
tipi_resp_cod='R0'
tipi_descrip='DESCRIPCION'
tipi_estado='tipo'
tipi_resp_estado='NO GESTIONADO'
tipi_subdescripcion='SUB_DESCRIPCION'
tnum_tb='tNumeroCenco_Sae'
tnum_dni='CODDOC'
tlista_generada='borrar_prestamo_cencosud'
get_base=since_base_maestra_cencosud_ppff

def resumen_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado):
    query = f"""
        SELECT *
        FROM OPENQUERY([192.168.3.{servidor_01}], '
            SELECT        
            rtrim(ltrim(d.vendor_lead_code)) AS vendor_lead_code,        
            e.dial_method,
            a.campaign_id AS numero_campana,        
            a.user AS dni_ejecutivo,
            c.full_name AS ejecutivo,
            e.campaign_name AS nombre_campana,        
            a.call_date AS fecha_hora_llamada,        
            a.length_in_sec AS duracion,        
            b.status_name AS call_result,        
            f.list_description,        
            f.list_name,        
            a.phone_number as phone_number,        
            d.alt_phone as fecha_agenda,        
            d.comments as comentarios,        
            a.status AS codigo,
            a.term_reason,	
            a.alt_dial
            FROM asterisk.vicidial_log a         
            LEFT JOIN asterisk.vicidial_list d ON a.lead_id=d.lead_id        
            LEFT JOIN asterisk.vicidial_campaigns e ON a.campaign_id=e.campaign_id        
            LEFT JOIN asterisk.vicidial_lists f ON a.list_id=f.list_id        
            LEFT JOIN asterisk.vicidial_statuses b ON a.status=b.status        
            LEFT JOIN asterisk.vicidial_users c ON a.user=c.user        
            WHERE (e.campaign_name like "%{tipi_cond1}" or e.campaign_name like "%{tipi_cond2}" or e.campaign_name like "%{tipi_cond3}")
            AND a.call_date >= DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01'')
            AND a.call_date < 
            DATE_ADD(DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01''), INTERVAL 1 MONTH)
        ')

        """
    df_vicidial=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

    df_vicidial = df_vicidial.withColumn(
        "vendor_lead_code",
        F.lpad(F.col("vendor_lead_code").cast("string"), 8, "0")
    )
    query = f"""
        SELECT {tipi_cod} as codigo
        , case
            when {tipi_cod}='CALLBK' then 'VOLVER A LLAMAR - call'
            else {tipi_descrip} 
        end as descripcion
        ,case 
            when {tipi_cod}='CALLBK' then 1200
            else peso 
        end as peso,
        tipo as estado  
        FROM [ODIN].[dbo].{tb_tipolofia}
        where LEFT({tipi_cod},2)='{tipi_resp_cod}' or {tipi_estado}='{tipi_resp_estado}' or {tipi_cod}='CALLBK'
        """
    df_tipi=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

    df_vicidial=df_vicidial.join(df_tipi,["codigo"],"left")

    return df_vicidial.select('fecha_hora_llamada','list_name','vendor_lead_code','phone_number','descripcion','dni_ejecutivo','ejecutivo','dial_method','term_reason','alt_dial','call_result','duracion','codigo','nombre_campana','estado','peso')

df_julio=resumen_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado)
# df_julio=df_julio.filter(F.col('fecha_hora_llamada')<'2026-07-13')

fecha_mes_base='2026-08-01'
df_agosto=resumen_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado)


In [3]:
query = f"""
    select a.Dni as CODDOC,b.Descripcion ,a.Inicio
    ,b.tipo as estado,b.PESO
    From SAMANTHA..tmp_llamadas_mes a
    left join ODIN.dbo.tTipologia_Cencosud_PPFF b
    on a.Codigo_Paleta=b.Codigo
    where a.Nombre_Campana in('2026-07 SAE','2026-07 SAE','AGENDA','2026-07 CD')
    AND a.Fecha_Llamada>='2026-07-01'
    AND a.Fecha_Llamada<='2026-07-17'
    """
df_julio=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
query = f"""
    select a.Dni as CODDOC,b.Descripcion ,a.Inicio
    ,b.tipo as estado,b.PESO
    From SAMANTHA..tmp_llamadas_mes a
    left join ODIN.dbo.tTipologia_Cencosud_PPFF b
    on a.Codigo_Paleta=b.Codigo
    where a.Nombre_Campana in('2026-08 SAE','2026-08 SAE','AGENDA')
    AND a.Fecha_Llamada>='2026-08-01'
    AND a.Fecha_Llamada<='2026-08-17'
    """
df_agosto=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)



In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_julio = df_julio.withColumn(
    "fecha",
    F.to_date("Inicio")
)
fechas_julio = (
    df_julio
    .select("fecha")
    .distinct()
    .orderBy("fecha")
)

fechas_julio = fechas_julio.withColumnRenamed(
    "fecha",
    "fecha_corte"
)

df_acumulado_julio = (
    df_julio
    .crossJoin(fechas_julio)
    .filter(
        F.col("fecha") <= F.col("fecha_corte")
    )
)

w = (
    Window
    .partitionBy(
        "fecha_corte",
        "CODDOC"
    )
    .orderBy(
        F.col("PESO").asc_nulls_last(),
        F.col("Inicio").desc_nulls_last()
    )
)
df_mejor_julio = (
    df_acumulado_julio
    .withColumn(
        "rn",
        F.row_number().over(w)
    )
    .filter(
        F.col("rn") == 1
    )
    .drop("rn")
)
df_mejor_julio=df_mejor_julio.withColumn('fec_ref',F.lit('2026-07-01'))

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_agosto = df_agosto.withColumn(
    "fecha",
    F.to_date("Inicio")
)
fechas_agosto = (
    df_agosto
    .select("fecha")
    .distinct()
    .orderBy("fecha")
)

fechas_agosto = fechas_agosto.withColumnRenamed(
    "fecha",
    "fecha_corte"
)

df_acumulado_agosto = (
    df_agosto
    .crossJoin(fechas_agosto)
    .filter(
        F.col("fecha") <= F.col("fecha_corte")
    )
)

w = (
    Window
    .partitionBy(
        "fecha_corte",
        "CODDOC"
    )
    .orderBy(
        F.col("PESO").asc_nulls_last(),
        F.col("Inicio").desc_nulls_last()
    )
)
df_mejor_agosto = (
    df_acumulado_agosto
    .withColumn(
        "rn",
        F.row_number().over(w)
    )
    .filter(
        F.col("rn") == 1
    )
    .drop("rn")
)
df_mejor_agosto=df_mejor_agosto.withColumn('fec_ref',F.lit('2026-08-01'))

In [6]:
df_mejor=df_mejor_agosto.unionByName(df_mejor_julio)

In [7]:
query = f"""
select CODDOC,PROPENSION,LINEA_SAE,TEA,PCT_SAE, PRODUCTO,
case
    when FECHA_ENVIO>='2026-08-01' then '2026-08-01'
    else '2026-07-01'
end as fec_ref
from DANTALION.dbo.Base_Maestra_Cencosud_PPFF
where FECHA_ENVIO>='2026-07-01'
    """
df_base=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)
# df_base.count()

['CODDOC', 'PROPENSION', 'LINEA_SAE', 'TEA', 'PCT_SAE', 'PRODUCTO', 'fec_ref']

In [8]:

df_base = (
    df_base
    .withColumn(
        "fec_ref",
        F.to_date("fec_ref")
    )
)

df_mejor = (
    df_mejor
    .withColumn(
        "fec_ref",
        F.to_date("fec_ref")
    )
    .withColumn(
        "fecha_corte",
        F.to_date("fecha_corte")
    )
)
fechas_corte = (
    df_mejor
    .select(
        "fec_ref",
        "fecha_corte"
    )
    .distinct()
)

In [73]:
base_diaria.columns


['CODDOC',
 'PROPENSION',
 'LINEA_SAE',
 'TEA',
 'PCT_SAE',
 'PRODUCTO',
 'fec_ref',
 'fecha_corte']

In [9]:
base_diaria = (
    df_base.alias("b")
    .join(
        fechas_corte.alias("f"),
        F.col("b.fec_ref") == F.col("f.fec_ref"),
        "inner"
    )
    .select(
        "b.*",
        F.col("f.fecha_corte")
    )
)

In [10]:
df_final = (
    base_diaria.alias("b")
    .join(
        df_mejor.alias("m"),
        (
            (F.col("b.CODDOC") == F.col("m.CODDOC"))
            &
            (F.col("b.fec_ref") == F.col("m.fec_ref"))
            &
            (F.col("b.fecha_corte") == F.col("m.fecha_corte"))
        ),
        "left"
    )
    .select(
        F.col("b.CODDOC"),
        F.col("b.PROPENSION"),
        F.col("b.PRODUCTO"),   
        F.col("b.LINEA_SAE"),
        F.col("b.TEA"),
        F.col("b.PCT_SAE"),
        F.col("b.fec_ref"),
        F.col("b.fecha_corte"),

        # datos provenientes de la gestión
        F.col("m.estado"),
        F.col("m.Descripcion")
    )
)

In [29]:
df_final.show(5)

+--------+----------+---------+-----+-------+----------+-----------+-------------+----+--------------------+
|  CODDOC|PROPENSION|LINEA_SAE|  TEA|PCT_SAE|   fec_ref|fecha_corte|       estado|PESO|         Descripcion|
+--------+----------+---------+-----+-------+----------+-----------+-------------+----+--------------------+
|06609513|     MEDIO|    30900|0.259|    024|2026-07-01| 2026-07-03|NO GESTIONADO|5101|MENSAJE EN CASILL...|
|06609513|     MEDIO|    30900|0.259|    024|2026-07-01| 2026-07-06|NO GESTIONADO|5101|MENSAJE EN CASILL...|
|06609513|     MEDIO|    30900|0.259|    024|2026-07-01| 2026-07-04|NO GESTIONADO|5101|MENSAJE EN CASILL...|
|06609513|     MEDIO|    30900|0.259|    024|2026-07-01| 2026-07-11|NO GESTIONADO|5101|MENSAJE EN CASILL...|
|06609513|     MEDIO|    30900|0.259|    024|2026-07-01| 2026-07-07|NO GESTIONADO|5101|MENSAJE EN CASILL...|
+--------+----------+---------+-----+-------+----------+-----------+-------------+----+--------------------+
only showing top 5 

In [11]:
from pyspark.sql import functions as F

df_final = (
    df_final
    .withColumn(
        "dia",
        F.dayofmonth("fecha_corte")
    )
)

In [12]:
resumen_diario = (
    df_final
    .groupBy(
        "fec_ref",
        "dia"
    )
    .agg(
        F.countDistinct("CODDOC").alias("base"),
        
        F.countDistinct(
            F.when(
                F.col("estado").isNotNull(),
                F.col("CODDOC")
            )
        ).alias("recorrido"),
        
        F.countDistinct(
            F.when(
                F.col("estado") == "CONTACTO EFECTIVO",
                F.col("CODDOC")
            )
        ).alias("contacto_efectivo")
    )
    .withColumn(
        "pct_recorrido",
        F.round(
            F.col("recorrido") / F.col("base") * 100,
            2
        )
    )
    .withColumn(
        "pct_contacto_efectivo",
        F.round(
            F.col("contacto_efectivo") / F.col("base") * 100,
            2
        )
    )
    .orderBy(
        "dia",
        "fec_ref"
    )
)

# resumen_diario.show(30, truncate=False)

In [13]:
resumen_resultado = (
    df_final
    .groupBy(
        "fec_ref",
        "dia",
        "estado",
        "PROPENSION",
        "PRODUCTO",
        "Descripcion"
    )
    .agg(
        F.countDistinct("CODDOC").alias("cantidad")
    )
    .orderBy(
        "dia",
        "fec_ref"
    )
)

In [14]:
df_resumen_pd = resumen_resultado.toPandas()

ruta_archivo = os.path.join(ruta_csv, 'cenco_mejor_tipi_dia1.xlsx')

df_resumen_pd.to_excel(ruta_archivo, index=False)


In [12]:
print([row['Descripcion' ] for row in df_agosto.select('Descripcion').distinct().collect()])


['Answering Machine Auto', 'Agenda', 'Agent Not Available', 'Busy Auto', 'Lead Being Called', 'No Answer AutoDial', 'Outbound Pre-Routing Drop', 'Disconnected Number Auto', None]


In [11]:
df_agosto.show(2)

+--------+--------------------+-------------------+------+----+
|  CODDOC|         Descripcion|             Inicio|estado|PESO|
+--------+--------------------+-------------------+------+----+
|72874933|Answering Machine...|2026-08-01 13:52:56|  NULL|NULL|
|07526999|Answering Machine...|2026-08-01 13:52:56|  NULL|NULL|
+--------+--------------------+-------------------+------+----+
only showing top 2 rows


In [6]:
df_julio.show(2)

+------------------+---------+----------------+------------+-----------+-------------+---------+-----------+-----------+--------+-----------+--------+------+--------------+------+----+
|fecha_hora_llamada|list_name|vendor_lead_code|phone_number|descripcion|dni_ejecutivo|ejecutivo|dial_method|term_reason|alt_dial|call_result|duracion|codigo|nombre_campana|estado|peso|
+------------------+---------+----------------+------------+-----------+-------------+---------+-----------+-----------+--------+-----------+--------+------+--------------+------+----+
+------------------+---------+----------------+------------+-----------+-------------+---------+-----------+-----------+--------+-----------+--------+------+--------------+------+----+



In [3]:
print(df_julio.columns)

['fecha_hora_llamada', 'list_name', 'vendor_lead_code', 'phone_number', 'descripcion', 'dni_ejecutivo', 'ejecutivo', 'dial_method', 'term_reason', 'alt_dial', 'call_result', 'duracion', 'codigo', 'nombre_campana', 'estado', 'peso']


In [ ]:
['fecha_hora_llamada', 'list_name', 'vendor_lead_code', 'phone_number', 'descripcion', 'dni_ejecutivo', 'ejecutivo', 'dial_method', 'term_reason', 'alt_dial', 'call_result', 'duracion', 'codigo', 'nombre_campana', 'estado']

In [ ]:
window_spec = Window.partitionBy(*cols_particion).orderBy(col("peso").asc_nulls_last(),col("fecha_hora_llamada").desc_nulls_last())

In [ ]:
df_julio.show()

In [ ]:
window_spec = Window.partitionBy(*cols_particion).orderBy(col("peso").asc_nulls_last(),col("fecha_hora_llamada").desc_nulls_last())
# window_spec = Window.partitionBy('vendor_lead_code','phone_number').orderBy(col("fecha_llamada").desc())
df_julio = df_julio.withColumn("ref_01", row_number().over(window_spec))
df_julio = df_julio.filter(col("ref_01") == 1).drop('ref_01')

In [ ]:
fecha_mes_base='2026-06-01'
fecha_mes_base_final='2026-06-01'
tipi_cond1='SAE'
tipi_cond2='SAR'
tipi_cond3='AGENDA'
tb_tipolofia='tTipologia_Cencosud_PPFF'
servidor_01=76
tipi_cod='Codigo'
tipi_resp_cod='R0'
tipi_descrip='DESCRIPCION'
tipi_estado='tipo'
tipi_resp_estado='NO GESTIONADO'
tipi_subdescripcion='SUB_DESCRIPCION'
tnum_tb='tNumeroCenco_Sae'
tnum_dni='CODDOC'
tlista_generada='borrar_prestamo_cencosud'
get_base=since_base_maestra_cencosud_ppff



In [ ]:
    fecha_mes_base='2026-06-01'
    tipi_cond1='DINERS TC'
    tipi_cond2='xx'
    tipi_cond3='xx'
    tb_tipolofia='tTipologia_Diners_TC'
    servidor_01=21
    tipi_cod='cod'
    tipi_resp_cod='E0'
    tipi_descrip='[NIVEL 4]'
    tipi_estado='[NIVEL 2]'
    tipi_resp_estado='NO CONTACTO'
    tipi_subdescripcion='[NIVEL 3]'
    tnum_tb='tNumeroDinersTc'
    tnum_dni='NUMERO_DOCUMENTO'
    tlista_generada='borrar_tc_dinner'
    get_base=since_base_maestra_tc_dinners

In [ ]:
df_sem1=lista_generada_dia(spark,'2026-07-01',tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado,tipi_subdescripcion,tnum_tb,tnum_dni,get_base,'2026-07-01')
df_sem2=lista_generada_dia(spark,'2026-07-01',tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado,tipi_subdescripcion,tnum_tb,tnum_dni,get_base,'2026-06-13')
df_sem3=lista_generada_dia(spark,'2026-07-11',tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado,tipi_subdescripcion,tnum_tb,tnum_dni,get_base,'2026-06-20')

In [ ]:
df_sem1=df_sem1.filter(F.col('fecha_envio')=='2026-06-01')
df_sem2=df_sem2.filter(F.col('fecha_envio')=='2026-06-01')
df_sem3=df_sem3.filter(F.col('fecha_envio')=='2026-06-01')


In [ ]:
df_sem1 = (
    df_sem1
    .withColumn("mejor_estado_tipi_cli", F.coalesce(F.col("mejor_estado_tipi_cli"), F.lit("MAQUINA")))
    .withColumn("mejor_descripcion_cli", F.coalesce(F.col("mejor_descripcion_cli"), F.lit("MAQUINA")))
)
df_sem2 = (
    df_sem2
    .withColumn("mejor_estado_tipi_cli", F.coalesce(F.col("mejor_estado_tipi_cli"), F.lit("MAQUINA")))
    .withColumn("mejor_descripcion_cli", F.coalesce(F.col("mejor_descripcion_cli"), F.lit("MAQUINA")))
)
df_sem3 = (
    df_sem3
    .withColumn("mejor_estado_tipi_cli", F.coalesce(F.col("mejor_estado_tipi_cli"), F.lit("MAQUINA")))
    .withColumn("mejor_descripcion_cli", F.coalesce(F.col("mejor_descripcion_cli"), F.lit("MAQUINA")))
)

In [ ]:
from pyspark.sql import functions as F

df_sem1 = (
    df_sem1
    .groupBy("mejor_estado_tipi_cli", "mejor_descripcion_cli")
    .agg(F.countDistinct("vendor_lead_code").alias("semana_1"))
)

df_sem2 = (
    df_sem2
    .groupBy("mejor_estado_tipi_cli", "mejor_descripcion_cli")
    .agg(F.countDistinct("vendor_lead_code").alias("semana_2"))
)

df_sem3 = (
    df_sem3
    .groupBy("mejor_estado_tipi_cli", "mejor_descripcion_cli")
    .agg(F.countDistinct("vendor_lead_code").alias("semana_3"))
)

In [ ]:
print(df_sem1.columns)
print(df_sem2.columns)
print(df_sem3.columns)

In [ ]:
df_resumen = (
    df_sem1
    .join(df_sem2, on=["mejor_estado_tipi_cli", "mejor_descripcion_cli"], how="outer")
    .join(df_sem3, on=["mejor_estado_tipi_cli", "mejor_descripcion_cli"], how="outer")
    .fillna(0)
)

In [ ]:

df_dni = df_resumen.toPandas()

ruta_archivo = os.path.join(ruta_csv, 'cenco_tipi.xlsx')

df_dni.to_excel(ruta_archivo, index=False)

In [ ]:
['vendor_lead_code', 'phone_number', 'title', 'first_name', 'last_name', 'address1', 'address2', 'address3', 'city', 'province', 'email', 'security_phrase', 'comments', 'mejor_codigo_telf', 'ult_codigo_cli', 'mejor_codigo_cli', 'indice_num', 'tipo_telf', 'indice_tpo_telf', 'indice', 'regimen_laboral', 'producto', 'marca', 'tea', 'retiro', 'fec_pago', 'fecha_ult_atm', 'monto_ant', 'fecha_ult_comp', 'monto_desembolsar', 'rng_saldo_tc_entre_linea_total_tc', 'disp_retiro_efect_', 'mejora_tasa', 'linea_sae', 'pct_sae', 'entidad1', 'deuda1', 'entidad2', 'deuda2', 'entidad3', 'deuda3', 'edad', 'frescura_target', 'flg_mejora', 'propension', 'tipdoc', 'fecha_envio', 'seg_monto', 'seg_tea', 'seg_edad', 'call_result', 'tramo', 'descripcion', 'peso', 'ult_call_result', 'fecha_llamada', 'fecha_hora_llamada', 'q_intentos', 'q_intentos_telf', 'q_intentos_dia', 'dni_unico', 'mejor_estado_tipi_cli', 'mejor_descripcion_cli', 'mejor_sub_descripcion', 'mejor_peso_cli', 'ult_estado_tipi_cli', 'ult_descripcion_cli', 'ult_sub_descripcion_cli', 'ult_peso_cli', 'mejor_estado_tipi_telf', 'mejor_descripcion_telf', 'mejor_sub_descripcion_telf', 'mejor_peso_telf']esta


### armar lista sae / sar

In [ ]:
df_sem1=df_sem1.withColumn('semana',F.lit(1))
df_sem2=df_sem2.withColumn('semana',F.lit(2))
df_sem3=df_sem3.withColumn('semana',F.lit(3))

In [ ]:
df_sem2=df_sem2.select('vendor_lead_code','semana')
df_sem3=df_sem3.select('vendor_lead_code','semana')

In [ ]:
df_sem1=df_sem1.join(df_sem2,['vendor_lead_code'],'left')
df_sem1=df_sem1.join(df_sem3,['vendor_lead_code'],'left')

In [ ]:


query = """
  SELECT * FROM CRONOX.dbo.borrar_prestamo_cencosud
  where fecha_envio='2026-06-01'
    """
df_list_cencosud = obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)

df_list_cencosud = df_list_cencosud.withColumn(
    'linea_sae',
    col('linea_sae').cast('int')
)
df_list_cencosud=df_list_cencosud.dropDuplicates(['vendor_lead_code'])

print(df_list_cencosud.count())
print(df_list_cencosud.columns)

In [ ]:
df_sem1=df_sem1.filter(F.col('fecha_envio')=='2026-06-01')

In [ ]:
df_sem1=df_sem1.dropDuplicates(['vendor_lead_code'])
# df_sem1.count()

In [ ]:
# df_list_cencosud=df_list_cencosud.select(
# 'vendor_lead_code','phone_number','title','first_name','last_name','address1','address2','address3','city','province','email','security_phrase','comments','mejor_codigo_cli','indice_num','tipo_telf','indice','regimen_laboral','producto','marca','tea','retiro','fec_pago','monto_ant','monto_desembolsar','rng_saldo_tc_entre_linea_total_tc','disp_retiro_efect_','mejora_tasa','linea_sae','pct_sae','edad','frescura_target','flg_mejora','propension','tipdoc','fecha_envio','seg_monto','seg_tea','seg_edad','fecha_llamada','fecha_hora_llamada','mejor_estado_tipi_cli','mejor_descripcion_cli','mejor_sub_descripcion')

In [ ]:
df_list_filtrada = df_sem1.toPandas()


In [ ]:
df_list_filtrada["fecha_llamada"] = pd.to_datetime(
    df_list_filtrada["fecha_llamada"],
    errors="coerce",
    dayfirst=True
)


In [ ]:
import locale

locale.setlocale(locale.LC_TIME, "Spanish_Peru.1252")
df_list_filtrada["fecha_llamada_bi"] = df_list_filtrada["fecha_llamada"].dt.strftime("%Y-%m-%d")

df_list_filtrada["dia"] = pd.to_datetime(df_list_filtrada["fecha_llamada"]).dt.day
df_list_filtrada["nombre_dia"] = pd.to_datetime(df_list_filtrada["fecha_llamada"]).dt.day_name()

df_list_filtrada["semana"] = pd.to_datetime(df_list_filtrada["fecha_llamada"]).dt.isocalendar().week
df_list_filtrada["semana_mes"] = (
    (pd.to_datetime(df_list_filtrada["fecha_llamada"]).dt.day - 1) // 7 + 1
)

In [ ]:
ruta_archivo = os.path.join(ruta_csv, 'semana_tipi.xlsx')
df_list_filtrada.to_excel(ruta_archivo, index=False)

In [ ]:
df_list_cencosud.dropDuplicates(['vendor_lead_code']).count()

In [ ]:
marca=[' N ', ' Y ', 'Y', 'N']


In [ ]:
# mejor_tipi=['VOLVER A LLAMAR - call',]
marca=['  ', ' Y ', 'Y',]

mejor_tipi=[
    'LLAMADA ELIMINADA POR ERROR EN RED (AUTO)',
 'VOLVER A LLAMAR',
 'CLIENTE CUELGA ANTES DE ESCUCHAR OFERTA',
 'NO UTILIZA (TARJETAS - PRESTAMOS)',
 'AUTODIAL NO RESPONDE (AUTO)',
 'OFERTA DE TASA MUY ALTA',
 'OFERTA DE LINEA MUY BAJA',
 'NO SE CONCRETO LLAMADA (CASILLA DE VOZ - APAGADO)',
 'SIN DESCRIPCION',
 'VOLVER A LLAMAR (TERCERO RELACIONADO)',
 'VOLVER A LLAMAR - call',
 'TELEFONO OCUPADO / NO CONTESTAN',
 'OCUPADO (AUTO)',
 'NO DESEA –NO ESPECIFICA MOTIVO',
 'GESTION EN PROCESO (AUTO)',
 'NUMERO DESCONECTADO (AUTO)',
 'MENSAJE EN CASILLA DE VOZ (AUTO)',
 'TELEFONO FUERA DE SERVICIO / NO EXISTE',
 'DESEA IR A AGENCIA',
 'AGENTE NO DISPONIBLE (AUTO)',
 ]
propension=['MEDIO', 'ALTO']


# df_filtrado = df_list_cencosud.filter(
#     # (F.col('title') == 'sae') &  
#     ((F.col('tipo_telf') == 'BBDD CEL01')&(F.length(F.col('phone_number')) == 9)) &
#     ((F.col('mejor_descripcion_cli').isin(mejor_tipi)) | (F.col('mejor_descripcion_cli').isNull())) &
#     ((F.col('mejor_descripcion_telf').isin(mejor_tipi)) | (F.col('mejor_CODIGO_telf').isNull()))&
#     ((F.col('fecha_llamada')<'2026-04-17') | (F.col('fecha_llamada').isNull())) &
#     # (F.col('linea_sae')>=7000)
    
# ).orderBy(col('linea_sae').desc())

# print(df_filtrado.count())

In [ ]:
df_filtrado = df_parte_3.select(
    'vendor_lead_code', 'phone_number', 'title', 'first_name',
    'last_name', 'address1', 'address2', 'address3',
    'city', 'province', 'email', 'security_phrase', 'comments',
)
df_dni = df_resumen.toPandas()

ruta_archivo = os.path.join(ruta_csv, 'df_parte_3.xlsx')

df_dni.to_excel(ruta_archivo, index=False)


In [ ]:

df_list_cencosud.groupBy('title') \
    .count() \
    .orderBy('title') \
    .show(30, truncate=False)

In [ ]:
print([row['mejor_descripcion_telf' ] for row in df_list_cencosud.select('mejor_descripcion_telf').distinct().collect()])
# mejor_estado_tipi_cli=['CONTVO', 'NCTO', 'CONTACTO EFECTIVO', 'ONADO',]


In [ ]:
df_list_filtrada = df_filtrado.select(
    'vendor_lead_code', 'phone_number', 'title', 'first_name',
    'last_name', 'address1', 'address2', 'address3',
    'city', 'province', 'email', 'security_phrase', 'comments'
).toPandas()
ruta_archivo = os.path.join(ruta_csv, 'rec.xlsx')
df_list_filtrada.to_excel(ruta_archivo, index=False)

In [ ]:
# mejor_tipi=['VOLVER A LLAMAR - call',]

mejor_descripcion_telf=[
    # 'VOLVER A LLAMAR',
 'CLIENTE CUELGA ANTES DE ESCUCHAR OFERTA',
 'NO UTILIZA (TARJETAS - PRESTAMOS)',
 'OFERTA DE TASA MUY ALTA',
 'OFERTA DE LINEA MUY BAJA',
 'NO SE CONCRETO LLAMADA (CASILLA DE VOZ - APAGADO)',
 'TELEFONO OCUPADO / NO CONTESTAN',
 'NO SE ASIGNO RESULTADO A LA LLAMADA (AUTO)',
 'NO DESEA –NO ESPECIFICA MOTIVO',
 'DESEA IR A AGENCIA',
 'CLIENTE DESEA OTRO PRODUCTO',
]


df_filtrado = df_list_cencosud.filter(
    ( 
        (F.col('mejor_estado_tipi_cli').isin('CONTACTO EFECTIVO')) | 
        (F.col('mejor_estado_tipi_cli').isNull())
    ) &     
    ( 
        (F.col('mejor_descripcion_cli').isin(mejor_descripcion_telf)) | 
        (F.col('mejor_descripcion_cli').isNull())
    ) &    
    ( 
        (F.col('mejor_descripcion_telf').isin(mejor_descripcion_telf)) | 
        (F.col('mejor_descripcion_telf').isNull())
    ) &
    (
        (F.col('tipo_telf')=='CEL01')|
        (F.col('tipo_telf')=='CEL02')|
        (F.col('tipo_telf')=='CEL03')
    )&
    # ((F.col('tipo_telf') == 'BBDD CEL01')&(F.length(F.col('phone_number')) == 9)) &
    # (F.col('mejor_estado_tipi_cli') != 'CONTACTO EFECTIVO') &  
    # (F.col('propension').isin('ALTO','MEDIO','')) &  
    # (F.col('linea_sae')>=10000) &  
        ((F.col('fecha_llamada')<'2026-06-06') | (F.col('fecha_llamada').isNull())) &

    # (
        
    #     (
    #         (F.col('fecha_llamada')=='2026-05-16') 
            #  (F.col('q_intentos_telf').isNull())&
    #      )
    # )

    (F.col('title') == 'REC') &  
    (F.col('retiro').isin('no_aplica')) 
    
).orderBy(col('linea_sae').desc())

df_filtrado=df_filtrado.dropDuplicates(['vendor_lead_code'])
print(df_filtrado.count())

In [ ]:
df_list_cencosud.filter(
    F.col('title') == 'sae'
).dropDuplicates(
    ['vendor_lead_code']
).count()

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy('propension',"entidad1",'seg_edad').orderBy(F.rand())

df_split = df_filtrado.withColumn("grupo_split", F.ntile(3).over(w))

df_parte_1 = df_split.filter(F.col("grupo_split") == 1).drop("grupo_split")
df_parte_2 = df_split.filter(F.col("grupo_split") == 2).drop("grupo_split")
df_parte_3 = df_split.filter(F.col("grupo_split") == 3).drop("grupo_split")
# df_parte_3 = df_split.filter(F.col("grupo_split") == 3).drop("grupo_split")
# df_parte_4 = df_split.filter(F.col("grupo_split") == 4).drop("grupo_split")
# df_parte_5 = df_split.filter(F.col("grupo_split") == 5).drop("grupo_split")
# df_parte_6 = df_split.filter(F.col("grupo_split") == 6).drop("grupo_split")
# df_parte_7 = df_split.filter(F.col("grupo_split") == 7).drop("grupo_split")

print(
    df_parte_1.count(),
    df_parte_2.count(),
    df_parte_3.count()
    # df_parte_4.count(),
    # df_parte_5.count(),
    # df_parte_6.count(),
    # df_parte_7.count()
)

In [ ]:

df_list_cencosud.groupBy('mejor_descripcion_cli') \
    .count() \
    .orderBy('mejor_descripcion_cli') \
    .show(30, truncate=False)

In [ ]:
query = """
  SELECT * FROM DANTALION.dbo.tNumeroCenco_Sae
    """
df_num = obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

In [ ]:
df_num.dropDuplicates(['CODDOC']).count()

In [ ]:
df_filtrado.dropDuplicates(['vendor_lead_code']) \
    .groupBy("propension") \
    .count() \
    .orderBy("propension") \
    .show()

In [ ]:
df_filtrado.groupBy("propension") \
    .count() \
    .orderBy("propension") \
    .show()

In [ ]:
df_filtrado

In [ ]:
tipificaicon_telf="mejor15_descripcion_telf"
pareto_mejor_descripcion(df_dni,tipificaicon_telf)
exportar_lista(df_filtrado,'SAR_202604021135.xlsx','linea_sae')

In [ ]:
cantidad_leads= df_filtrado.count()

In [ ]:
info_list=f"""
Se tienen {cantidad_leads} leads
maximo de intentos por num telef: 20
montos menores a 9k
propension ALTO, MEDIO
"""

print(info_list)

In [ ]:
exportar_lista(df_filtrado,'sae_202603281246.xlsx','linea_sae')

In [ ]:
print([row['tipo_telf' ] for row in df_list_cencosud.select('tipo_telf').distinct().collect()])


In [ ]:
['LLAMADA ELIMINADA POR ERROR EN RED (AUTO)', 'VOLVER A LLAMAR', 'CLIENTE CUELGA ANTES DE ESCUCHAR OFERTA', 'NO UTILIZA (TARJETAS - PRESTAMOS)', 'AUTODIAL NO RESPONDE (AUTO)', 'OFERTA DE TASA MUY ALTA', 'OFERTA DE LINEA MUY BAJA', 'NO DESEA PAGAR MEMBRESIA', 'NO SE CONCRETO LLAMADA (CASILLA DE VOZ - APAGADO)', 'VOLVER A LLAMAR (TERCERO RELACIONADO)', 'ZONA FUERA DE COBERTURA', 'NO VOLVER A LLAMAR NUNCA MAS', 'VOLVER A LLAMAR - call', 'TELEFONO OCUPADO / NO CONTESTAN', 'OCUPADO (AUTO)', 'NO SE ASIGNO RESULTADO A LA LLAMADA (AUTO)', 'NO DESEA –NO ESPECIFICA MOTIVO', 'TELEFONO EQUIVOCADO', 'NUMERO DESCONECTADO (AUTO)', 'MENSAJE EN CASILLA DE VOZ (AUTO)', 'CLIENTE FALLECIO', 'CLIENTE ACEPTA PRODUCTO', 'TELEFONO FUERA DE SERVICIO / NO EXISTE', 'DESEA IR A AGENCIA', 'CLIENTE DESEA OTRO PRODUCTO', 'AGENTE NO DISPONIBLE (AUTO)', None]


In [ ]:
import os

ruta_archivo = os.path.join(ruta_csv, 'sae_202604021110.xlsx')

df_dni.to_excel(ruta_archivo, index=False)
df_dni.count()

In [ ]:
df_dni.count()


In [ ]:

df_dni = df_filtrado.toPandas()


In [ ]:
los pu pm los qu epodemos usar


In [ ]:

nombre_archivo_xlsx='SAE_202603251722.xlsx'
columna_ref='linea_sae'
exportar_lista(df_filtrado,nombre_archivo_xlsx,columna_ref)


In [ ]:
fecha_mes_base='2026-03-01'
t_maestra='Base_Maestra_Cencosud_PPFF'
t_name_vigente='Base_Maestra_Cencosud_PPFF_vigente'
t_mumeros='tNumeroCenco_Sae'
cols_drop=[ 'FRESCURA_TARGET', 'REP1', 'REP2', 'REP3','FLAT2']


### armar lista negocios

In [ ]:
print(df_list.columns)

In [ ]:
df_dni.count()

In [ ]:
df_prueba.show()

In [ ]:
df_list=df_list.join(df_list,['vendor_lead_code','phone_number'],'leftanti')

In [ ]:
df_list.count()

In [ ]:
mejor_tipi_cet=['NO QUIERE', 'VOLVER A LLAMAR', 'LO VA A PENSAR', 'DESEA ELECTRO', 'MENSAJES CON TERCEROS - HIJOS', 'TELEFONO APAGADO', 'MENSAJES CON TERCEROS - ESPOSA', 'OTROS', 'NO CONTESTA ', 'MENSAJES CON TERCEROS - OTROS', 'CLIENTE DE VIAJE', 'TONO OCUPADO', 'DESEA MAS MONTO']


In [ ]:
info_list=f"""
Se tienen {cantidad_leads} leads
Zonas : {zona_list}
cet: {mejor_tipi_cet}
{tip_prioridad_list}
{email_list}
"""

print(info_list)

In [ ]:
df_filtrado = df_filtrado.select(
    'vendor_lead_code', 'phone_number_02', 'title', 'first_name',
    'last_name', 'address1', 'address2', 'address3',
    'city', 'province', 'email', 'security_phrase', 'comments'
)
df_dni = df_filtrado.toPandas()

In [ ]:
import matplotlib.pyplot as plt

df_dni['email'].value_counts().plot(kind='bar')

plt.title('Top departamentos (email)')
plt.xlabel('Departamento')
plt.ylabel('Cantidad')
plt.xticks(rotation=45)
plt.show()

In [ ]:

import os

ruta_archivo = os.path.join(ruta_csv, 'negocio_202603271247.xlsx')

df_dni.to_excel(ruta_archivo, index=False)

In [ ]:
df_list.select('retiro_01').filter(col('retiro_01')==0).count()

In [ ]:
# print([row['first_name'] for row in df_list.select('first_name').distinct().collect()])
# print([row['title'] for row in df_list.select('title').distinct().collect()])
# print([row['tip_prioridad'] for row in df_list.select('tip_prioridad').distinct().collect()])
# print([row['email'] for row in df_list.select('email').distinct().collect()])
# print([row['prioridada'] for row in df_list.select('prioridada').distinct().collect()])
print([row['mejor15_descripcion_telf' ] for row in df_list.select('mejor15_descripcion_telf').distinct().collect()])
# print(df_list.columns)


## inventario

In [ ]:
fecha_ref = '2026-04-01'

campana_name_1='SAE'
campana_name_2='SAR'
campana_name_3='AGENDA'
venta_campana='Cenco_Prest'
tb_tipologia='tTipologia_Cencosud_PPFF'
dni_tnumero='CODDOC'
cod_letra='R'
tb_tnumero='tNumeroCenco_Sae'

query = f"""
SELECT dni as vendor_lead_code,
phone_number,
codigo_paleta,
    DATEDIFF(MONTH, fecha_llamada, cast('{fecha_ref}'as date)) AS n_mes
FROM SAMANTHa.dbo.tmp_llamadas_mes 
WHERE (nombre_campana LIKE '%{campana_name_1}' or nombre_campana LIKE '%{campana_name_2}' or nombre_campana LIKE '%{campana_name_3}')
AND fecha_llamada >= DATEADD(MONTH, -6, cast('{fecha_ref}'as date))
AND fecha_llamada <  cast('{fecha_ref}'as date)
"""
df_llamadas=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

query = f"""
    select 
    DNI as vendor_lead_code,1 as venta
    from SAMANTHA.dbo.Ventas_Target
    where campana='{venta_campana}'
    AND cast(fecha as date) >= DATEADD(MONTH, -2, cast('{fecha_ref}'as date))
    AND cast(fecha as date) <  cast('{fecha_ref}'as date)
    """
df_venta= obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

query = f"""
select CODIGO as codigo_paleta,peso,descripcion from ODIN.dbo.{tb_tipologia}
where TIPO='CONTACTO EFECTIVO'
and LEFT(CODIGO,1)='{cod_letra}'
"""
df_tipi = obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

df_llamadas=df_llamadas.join(df_venta,['vendor_lead_code'],'leftanti')
df_llamadas=df_llamadas.join(df_tipi,['codigo_paleta'],'inner')

window_spec = Window.partitionBy("vendor_lead_code",'phone_number').orderBy(F.col("n_mes").asc_nulls_last(),F.col("peso"))
df_llamadas = df_llamadas.withColumn("ref_01", row_number().over(window_spec))
df_llamadas = df_llamadas.filter(F.col("ref_01") == 1).drop('ref_01')

query = f"""
select distinct {dni_tnumero} as vendor_lead_code,Telefonos as phone_number, 1 as ref 
from DANTALION.dbo.{tb_tnumero}
"""
df_tnumero=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)
df_tnumero.columns
df_tnumero=df_tnumero.join(df_llamadas,['vendor_lead_code', 'phone_number'],'inner')



In [ ]:


df_tnumero=df_tnumero.withColumn('n_lista',when(((col('descripcion')=='SI QUIERE')|(col('descripcion')=='CREDITO CONCRETADO'))&(col('n_mes')==1),1)
                                            .when((col('descripcion').isin(tipis))&(col('n_mes')>1),3)
                                            .when((col('descripcion').isin(tipis))&(col('n_mes')==1),4)
                                            .when((col('descripcion').isin(tipis))&(col('n_mes')==1),4)
                                            .otherwise(0))

overwrite_table_SQL(spark,df_tnumero,'cet_efe_negocios',server_sa,user_sa,pwd_sa,'CRONOX')

In [ ]:

fecha_ref = '2026-04-01'

lla_nombre_campana1='SAE'
lla_nombre_campana2='SAR'
lla_nombre_campana3='AGENDA'
venta_campana='Efectiva'
tb_tipologia='tTipologia_Cencosud_PPFF'
dni_tnumero='CODDOC'
cod_letra='BR'
tb_tnumero='tNumeroCenco_Sae'

query = f"""
SELECT dni as vendor_lead_code,
phone_number,
codigo_paleta,
    DATEDIFF(MONTH, fecha_llamada, '{fecha_ref}') AS n_mes
FROM SAMANTHa.dbo.tmp_llamadas_mes 
WHERE (nombre_campana LIKE '%{lla_nombre_campana1}' | nombre_campana LIKE '%{lla_nombre_campana2}' | nombre_campana LIKE '%{lla_nombre_campana3}')
AND fecha_llamada >= DATEADD(MONTH, -6, '{fecha_ref}')
AND fecha_llamada <  '{fecha_ref}'
"""
df_llamadas=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

# query = f"""
#     select 
#     DNI as vendor_lead_code,1 as venta
#     from SAMANTHA.dbo.Ventas_Target
#     where campana='{venta_campana}'
#     and cast(fecha as date) between '2026-02-01' and EOMONTH('{fecha_ref}')
#     """
# df_venta= obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)


In [ ]:

query = f"""
select CODIGO as codigo_paleta,peso,descripcion from ODIN.dbo.{tb_tipologia}
where TIPO='CONTACTO EFECTIVO'
and LEFT(CODIGO,1)='{cod_letra}'
"""
df_tipi = obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

# df_llamadas=df_llamadas.join(df_venta,['vendor_lead_code'],'leftanti')
df_llamadas=df_llamadas.join(df_tipi,['codigo_paleta'],'inner')

window_spec = Window.partitionBy("vendor_lead_code",'phone_number').orderBy(F.col("n_mes").asc_nulls_last(),F.col("peso"))
df_llamadas = df_llamadas.withColumn("ref_01", row_number().over(window_spec))
df_llamadas = df_llamadas.filter(F.col("ref_01") == 1).drop('ref_01')

query = f"""
select distinct {dni_tnumero} as vendor_lead_code,Telefonos as phone_number, 1 as ref 
from DANTALION.dbo.{tb_tnumero}
"""
df_tnumero=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)
df_tnumero.columns
df_tnumero=df_tnumero.join(df_llamadas,['vendor_lead_code', 'phone_number'],'inner')





# historico propension

In [ ]:
fecha_mes_base='2026-04-01'
t_maestra=tb_maestra_cenco_ppff
tnum_dni='CODDOC'
prioridad='propension'

query = f"""
SELECT 
vendor_lead_code,
prioridad_hist
FROM (
    SELECT 
        RIGHT(REPLICATE('0',8) + CAST(a.{tnum_dni} AS VARCHAR(20)), 8) as vendor_lead_code,
        b.{prioridad} as prioridad_hist,
        TRY_CONVERT(date, b.fecha_envio) as fecha_envio,
        ROW_NUMBER() OVER (
            PARTITION BY a.{tnum_dni} 
            ORDER BY TRY_CONVERT(date, b.fecha_envio) DESC
        ) as rn
    FROM DANTALION.dbo.{t_maestra} a
    INNER JOIN DANTALION.dbo.{t_maestra} b
        ON a.{tnum_dni} = b.{tnum_dni}
    WHERE b.{prioridad} IS NOT NULL
    AND b.{tnum_dni} IS NOT NULL
    AND TRY_CONVERT(date, b.fecha_envio) IS NOT NULL
    AND TRY_CONVERT(date, b.fecha_envio) >= DATEADD(MONTH, -6, '{fecha_mes_base}')
    AND TRY_CONVERT(date, b.fecha_envio) <'{fecha_mes_base}'
) t
WHERE rn = 1
"""
df_hist=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)
overwrite_table_SQL(spark,df_hist,'borrar_pp_cenco_prioridad_hist',server_sa,user_sa,pwd_sa,'CRONOX')
print(df_hist.columns)

In [ ]:

list_tipi=[
    'VOLVER A LLAMAR',
     'CLIENTE CUELGA ANTES DE ESCUCHAR OFERTA',
     'NO UTILIZA (TARJETAS - PRESTAMOS)',
     'OFERTA DE TASA MUY ALTA',
     'OFERTA DE LINEA MUY BAJA',
     'NO DESEA PAGAR MEMBRESIA',
     'NO DESEA –NO ESPECIFICA MOTIVO',
     'CLIENTE ACEPTA PRODUCTO',
     'DESEA IR A AGENCIA',
     'CLIENTE DESEA OTRO PRODUCTO'
]
fecha_mes_base = '2026-04-01'

tipi_cond1='SAE'
tipi_cond2='SAR'
tipi_cond3='AGENDA'
tipi_cod='Codigo'

tipi_resp_cod='R'
tipi_descrip='DESCRIPCION'
tipi_estado='tipo'

tb_tipologia='tTipologia_Cencosud_PPFF'
dni_tnumero='CODDOC'
tb_tnumero='tNumeroCenco_Sae'

query = f"""
SELECT dni as vendor_lead_code,
phone_number,
codigo_paleta,
    DATEDIFF(MONTH, fecha_llamada, '{fecha_mes_base}') AS n_mes
FROM SAMANTHa.dbo.tmp_llamadas_mes 
WHERE dni is not null 
and (nombre_campana LIKE '%{tipi_cond1}' or nombre_campana LIKE '%{tipi_cond2}' or nombre_campana LIKE '%{tipi_cond3}')
AND fecha_llamada >= DATEADD(MONTH, -6, '{fecha_mes_base}')
AND fecha_llamada <  '{fecha_mes_base}'
"""
df_llamadas=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

query = f"""
select {tipi_cod} as codigo_paleta,peso,{tipi_descrip} as descripcion from ODIN.dbo.{tb_tipologia}
where {tipi_estado}='CONTACTO EFECTIVO'
and LEFT({tipi_cod},1)='{tipi_resp_cod}'
"""
df_tipi = obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

In [ ]:


list_tipi=[
    'VOLVER A LLAMAR',
     'CLIENTE CUELGA ANTES DE ESCUCHAR OFERTA',
     'NO UTILIZA (TARJETAS - PRESTAMOS)',
     'OFERTA DE TASA MUY ALTA',
     'OFERTA DE LINEA MUY BAJA',
     'NO DESEA PAGAR MEMBRESIA',
     'NO DESEA –NO ESPECIFICA MOTIVO',
     'CLIENTE ACEPTA PRODUCTO',
     'DESEA IR A AGENCIA',
     'CLIENTE DESEA OTRO PRODUCTO'
]
df_tipi=df_tipi.filter(col('descripcion').isin(list_tipi))
df_llamadas=df_llamadas.join(df_tipi,['codigo_paleta'],'inner')

window_spec = Window.partitionBy("vendor_lead_code", "phone_number") \
    .orderBy(
        F.col("n_mes").asc_nulls_last(),
        F.col("peso").desc_nulls_last()
    )

df_llamadas = df_llamadas.withColumn("ref_01", row_number().over(window_spec))
df_llamadas = df_llamadas.filter(F.col("ref_01") == 1).drop('ref_01')

query = f"""
select distinct {dni_tnumero} as vendor_lead_code,Telefonos as phone_number 
from DANTALION.dbo.{tb_tnumero}
"""
df_tnumero=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)
df_tnumero=df_tnumero.join(df_llamadas,['vendor_lead_code', 'phone_number'],'inner')
overwrite_table_SQL(spark,df_tnumero,'borrar_pp_cenco_cet',server_sa,user_sa,pwd_sa,'CRONOX')

In [ ]:
print([row['descripcion' ] for row in df_tipi.select('descripcion').distinct().collect()])


In [ ]:
df_tnumero.count()

In [ ]:
fecha_ref = '2026-04-01'

lla_nombre_campana1='SAE'
lla_nombre_campana2='SAR'
lla_nombre_campana3='AGENDA'
venta_campana='Efectiva'
tb_tipologia='tTipologia_Cencosud_PPFF'
dni_tnumero='CODDOC'
cod_letra='BR'
tb_tnumero='tNumeroCenco_Sae'

query = f"""
SELECT dni as vendor_lead_code,
phone_number,
codigo_paleta,
    DATEDIFF(MONTH, fecha_llamada, '{fecha_ref}') AS n_mes
FROM SAMANTHa.dbo.tmp_llamadas_mes 
WHERE (nombre_campana LIKE '%{lla_nombre_campana1}' | nombre_campana LIKE '%{lla_nombre_campana2}' | nombre_campana LIKE '%{lla_nombre_campana3}')
AND fecha_llamada >= DATEADD(MONTH, -6, '{fecha_ref}')
AND fecha_llamada <  '{fecha_ref}'
"""
df_llamadas=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)